# Diabetes Prediction

**Tabular Classification Project** — Predict diabetes using Pima Indians dataset.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Pima Indians Diabetes (768 rows × 9 columns, local `diabetes.csv`)  
Target: `Outcome` (0/1)

## 2 · Project Overview

This notebook focuses on **predicting diabetes** from clinical measurements using modern gradient-boosting models. We emphasize prediction quality, proper evaluation, and comparison of multiple approaches. The local CSV `diabetes.csv` contains the same Pima Indians dataset.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Build a complete diabetes prediction pipeline.
2. Handle medical data quality issues (zero-encoded missing values).
3. Compare automated baselines (LazyPredict, FLAML) with manually tuned boosting models.
4. Evaluate healthcare classification with appropriate metrics.
5. Understand precision-recall tradeoffs in medical diagnosis.

## 4 · Problem Statement

Predict whether a patient has diabetes based on 8 diagnostic measurements. Priority is high **recall** for diabetic patients.

## 5 · Why This Project Matters

- Diabetes affects 10%+ of adults globally — early prediction enables intervention.
- Medical ML projects require careful handling of data quality and evaluation metrics.
- This is a gateway project for healthcare ML.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 768 |
| **Columns** | 9 |
| **Source** | Local `diabetes.csv` |
| **Target** | Outcome (0/1) |
| **Known issues** | Zeros encode missing values in 5 columns |

## 7 · Dataset Source and License Notes

- UCI ML Repository / NIDDK.
- Public domain for educational use.
- Local copy: `diabetes.csv`.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, i in [("catboost",None),("lightgbm",None),("xgboost",None),("flaml",None),("lazypredict",None)]:
    _install(p, i)
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, re, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
np.random.seed(SEED)
print("Core imports loaded.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Outcome"
TEST_SIZE = 0.2
LOCAL_CSV = os.path.join(SAVE_DIR, "diabetes.csv")
print(f"Target: {TARGET}, CSV: {LOCAL_CSV}")

## 11 · Dataset Download or Loading

In [ ]:
if os.path.exists(LOCAL_CSV):
    df = pd.read_csv(LOCAL_CSV)
    print(f"Loaded from local: {LOCAL_CSV}")
else:
    import kagglehub
    path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
    import glob
    csv_files = glob.glob(os.path.join(path, "*.csv"))
    df = pd.read_csv(csv_files[0])
    print("Loaded from kagglehub")

print(f"Shape: {df.shape}")
print(df.head())

## 12 · Data Validation Checks

In [ ]:
print(f"Nulls:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")

zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nZeros (=missing):")
for c in zero_cols:
    n = (df[c] == 0).sum()
    print(f"  {c}: {n} ({n/len(df)*100:.1f}%)")

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
num_cols = [c for c in df.columns if c != TARGET]
for i, c in enumerate(num_cols):
    ax = axes[i // 4, i % 4]
    df[c].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(c)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_correlation.png"), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(f"Target distribution:\n{df[TARGET].value_counts()}")
print(f"\nProportions:\n{df[TARGET].value_counts(normalize=True)}")
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue", "coral"])
ax.set_title("Outcome Distribution")
ax.set_xticklabels(["No Diabetes", "Diabetes"], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_target.png"), dpi=100)
plt.show()

## 15 · Train/Validation/Test Split Strategy

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_train.columns]
X_test.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_test.columns]
print(f"NaN remaining: {X_train.isnull().sum().sum()}")

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part["Glucose_BMI"] = df_part["Glucose"] * df_part["BMI"]
    df_part["Age_Pregnancies"] = df_part["Age"] * df_part["Pregnancies"]
    df_part["Insulin_Glucose_Ratio"] = df_part["Insulin"] / (df_part["Glucose"] + 1)
print(f"Features: {X_train.shape[1]}")

## 18 · Baseline Model

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
baseline.fit(X_train, y_train)
bl_pred = baseline.predict(X_test)
print("Baseline — Logistic Regression:")
print(f"  Accuracy: {accuracy_score(y_test, bl_pred):.4f}")
print(f"  F1:       {f1_score(y_test, bl_pred, average='weighted'):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lp = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
lp_models, lp_preds = lp.fit(X_train, X_test, y_train, y_test)
print(lp_models.head(20))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task="classification", time_budget=60,
               metric="accuracy", verbose=0, seed=SEED)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy : {accuracy_score(y_test, flaml_pred):.4f}")
    print(f"  F1       : {f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## 21 · Additional Best-Library Workflow

## 22 · Top 3 Model Selection

## 23 · Final Training and Evaluation of Top 3

In [ ]:
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

import torch
device = "gpu" if torch.cuda.is_available() else "cpu"
xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
n_pos = int(y_train.sum()); n_neg = int(len(y_train) - n_pos); spw = n_neg / max(n_pos, 1)

models = {
    "CatBoost": CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                                    auto_class_weights="Balanced",
                                    task_type="GPU" if device=="gpu" else "CPU",
                                    verbose=0, random_seed=SEED),
    "LightGBM": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                    is_unbalance=True, device=device,
                                    verbose=-1, random_state=SEED),
    "XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                  scale_pos_weight=spw, device=xgb_device,
                                  eval_metric="logloss", verbosity=0, random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    results[name] = {"accuracy": acc, "f1": f1}
    print(f"{name}: Acc={acc:.4f} F1={f1:.4f}")

res_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nRanking:")
print(res_df)

## 24 · Error Analysis

In [ ]:
best_name = res_df.index[0]
best_model = models[best_name]
preds_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(y_test, preds_best))

errors = X_test.copy()
errors["true"] = y_test.values
errors["pred"] = preds_best
errors["wrong"] = errors["true"] != errors["pred"]
print(f"Total errors: {errors['wrong'].sum()} / {len(errors)} ({errors['wrong'].mean()*100:.1f}%)")
print("\nSample misclassifications:")
print(errors[errors["wrong"]].head(10))

## 25 · Interpretation and Business Insight

- Glucose level is the single strongest predictor — consistent with medical knowledge.
- Boosting models handle feature interactions and imputed values well.
- Clinical deployment should prioritize recall for positive cases.

## 26 · Limitations

- Small dataset (768 rows) — estimates have high variance.
- Only one ethnic group represented.
- Zero-value imputation introduces noise.

## 27 · How to Improve This Project

1. Use cross-validation for stability.
2. Try TabPFN-v2 for small-data advantage.
3. SHAP analysis for explanations.
4. Probability calibration for clinical use.

## 28 · Production Considerations

- Calibrated probabilities; HIPAA compliance; audit trails; monitoring.

## 29 · Common Mistakes

1. Not handling zero-encoded missing values.
2. Leaking via pre-split imputation.
3. Using accuracy alone on imbalanced data.
4. Not stratifying splits.

## 30 · Mini Challenge / Exercises

1. Run 5-fold CV and report mean ± std.
2. Add a Random Forest to the top 3 comparison.
3. Create precision-recall curves for threshold selection.

## 31 · Final Summary / Key Takeaways

In [ ]:
metrics = {}
for name, m in results.items():
    metrics[name] = m
best = res_df.index[0]
metrics["best_model"] = best
metrics["best_accuracy"] = results[best]["accuracy"]
metrics["best_f1"] = results[best]["f1"]

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))
print("\n✅ Diabetes Prediction notebook complete.")